**Human Movement Intelligence | Multimodal Movement Learning | Presented by: Reza Saadatyar (2026)**

**04 | Participant-Grouped Multimodal Fusion and Sensor Perturbation**

![Matched sEMG–acceleration inputs, independent fusion models and participant-level Development evaluation](../../images/04_fusion_and_modality_ablations.svg)

**Research question:** Does source-aligned acceleration improve movement recognition over the strongest unimodal method when both are assessed on the same unseen Development participants?

**Inputs:** Notebook 02's fixed E1 paired windows and participant-disjoint folds, and Notebook 03's matching original out-of-fold class probabilities.

**Methods:** Matched early feature fusion, fixed-weight classical/neural decision fusion and separately trained CNN, TCN, gated and compact cross-attention fusion. Synthetic acceleration replacements probe perturbation sensitivity, not real sensor failures.

**Primary metric:** Fixed-class macro F1 per unseen Development participant, averaged equally across people. All Test participants remain reserved for Notebook 05.


**00. Colab Repository Setup**


In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**


In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("multimodal_movement_learning")

# ---------------------------- 2. Standard Library ----------------------------
from pathlib import Path

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

plot_style(
    font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(9, 4),
)

# ------------------ 5. Deep Learning and Compute Device ----------------------
from core.computer_device import computer_device
device = computer_device(min_vram_gb=None)

# ------------------------ 6. Research Model Utilities ------------------------
from ninapro_dataset import ensure_numerical_cache
from ninapro_evaluation import select_model
from ninapro_models import FUSION_MODELS, UNIMODAL_MODELS
from ninapro_fusion_experiments import (
    load_unimodal_oof, run_early_fusion_arm, run_fixed_late_arm,
    run_neural_fusion_arm, consolidate_fusion_outputs,
    evaluate_fusion_finalist_seeds, PERTURBATIONS,
)

# --------------------- 7. Canonical Study File Locations ---------------------
study_dir = root / "multimodal_movement_learning"
output_dir = study_dir / "output"
processed_dir = root / "data" / "processed" / "ninapro_db2"
output_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
print("Research output directory:", output_dir)


**02. Read Matched Development Windows and Notebook 03 Evidence**

Reuse Notebook 02's original source-window IDs, corrected classes and participant-disjoint Development folds. The five reserved Test people remain outside fitting and model selection.

Notebook 03's *real* out-of-fold soft probabilities are required for decision-level fusion. Since Git ignores NPZ files, the output ZIP from Notebook 03 can supply the original probabilities. The input check validates identities, fold assignments, true labels and predicted classes against the published keyed prediction CSV files. Hard labels are not converted into artificial probabilities.


In [ ]:
# =========== 02. Read Matched Development and Unimodal Evidence ==============

# -------------------- 1. Fixed Source and Label Definitions -----------------
NOTEBOOK_NUMBER = 4
SEED = 42
index = pd.read_csv(output_dir / "02_window_index.csv")
roles = pd.read_csv(output_dir / "02_participant_split_assignments.csv")
folds = pd.read_csv(output_dir / "02_development_fold_assignments.csv")
classes = np.sort(
    pd.read_csv(output_dir / "01_label_dictionary.csv").label.to_numpy(int)
)
index["role"] = index.subject_id.map(
    dict(zip(roles.subject_id, roles.role))
)
index["validation_fold"] = index.subject_id.map(
    dict(zip(folds.subject_id, folds.validation_fold))
)
if index.role.isna().any():
    raise ValueError("Missing frozen Development/Test participant assignment.")
dev = index.loc[index.role == "Development"].copy()
test = index.loc[index.role == "Test"].copy()
if dev.empty or test.empty:
    raise ValueError("Development and reserved Test participants are required.")
if set(dev.subject_id) & set(test.subject_id):
    raise ValueError("Development and Test participants overlap.")
if dev.validation_fold.isna().any() or not index.window_key.is_unique:
    raise ValueError("Participant-fold or paired-window identity mismatch.")
common_folds = sorted(dev.validation_fold.unique().astype(int).tolist())

# ------------------ 2. Notebook 03 Out-of-Fold Probabilities ------------------
unimodal_metrics = pd.read_csv(output_dir / "03_all_unimodal_metrics.csv")
previous = load_unimodal_oof(index, dev, classes, output_dir)
print(
    "Development participants:", dev.subject_id.nunique(),
    "| paired windows:", len(dev),
)
print("Validation people by fold:")
display(dev.groupby("validation_fold").subject_id.nunique().rename("participants"))
print("Reserved Test participants:", test.subject_id.nunique())


**03. Recover Paired Statistical and Temporal Representations**

Classical feature fusion concatenates 48 sEMG and 96 acceleration features from the same source-window identity. Neural fusion reads corresponding 12-channel sEMG and 36-column acceleration recordings from the released common time grid.

Existing compatible numerical inputs are reused; missing inputs are retrieved and constructed. Scaling and neural normalisation use Training participants only, while identifiers, corrected labels and repetition metadata are never predictive features.


In [ ]:
# ========= 03. Recover Source-Aligned Features and Temporal Inputs ===========

emg_features, acc_features = ensure_numerical_cache(
    root, index, allow_source_download=True, verbose=True,
)
if not (
    len(emg_features) == len(acc_features) == len(index)
    and index.window_key.is_unique
):
    raise ValueError("Paired numerical inputs do not match Notebook 02.")
print("Early-fusion feature width:", emg_features.shape[1] + acc_features.shape[1])


**04. F1: Early-Fusion Logistic Regression**

A 144-dimensional paired feature vector combines the same 48 sEMG and 96 acceleration statistics per window. Scaling is fitted within each Training fold; all seven held-out people in the matching Validation fold contribute to macro F1.


In [ ]:
# ================= 04. F1: Early-Fusion Logistic Regression ==================

# --------------------- 1. Participant-Grouped Model Fit ----------------------
display(run_early_fusion_arm(
    "F1_logreg", dev, emg_features, acc_features, classes,
    output_dir=output_dir, folds=common_folds, seed=SEED,
))


**05. F1: Early-Fusion Random Forest**

A 120-tree class-weighted Random Forest uses the same paired feature vector. Its five participant-disjoint folds, class definitions and feature window identities match those of Logistic Regression.


In [ ]:
# ==================== 05. F1: Early-Fusion Random Forest =====================

# --------------------- 1. Participant-Grouped Model Fit ----------------------
display(run_early_fusion_arm(
    "F1_rf", dev, emg_features, acc_features, classes,
    output_dir=output_dir, folds=common_folds, seed=SEED,
))


**06. F2: Classical Logistic Regression Late Fusion**

The 0.5/0.5 decision average combines the independently obtained, class-aligned Notebook 03 sEMG and acceleration Logistic Regression probabilities. No outer Validation labels are used to fit or select the mixing weights.


In [ ]:
# ============= 06. F2: Classical Logistic Regression Late Fusion =============

# --------------------- 1. Participant-Grouped Model Fit ----------------------
display(run_fixed_late_arm(
    "F2_logreg", index, dev, previous, classes,
    output_dir=output_dir, folds=common_folds,
    emg_name="B1_logreg", acc_name="B2_logreg", seed=SEED,
))


**07. F2: Classical Random Forest Late Fusion**

The Random Forest decision-level control averages the sEMG and acceleration Random Forest OOF probabilities with predeclared equal weights, preserving the original 18 source-class columns.


In [ ]:
# ================ 07. F2: Classical Random Forest Late Fusion ================

# --------------------- 1. Participant-Grouped Model Fit ----------------------
display(run_fixed_late_arm(
    "F2_rf", index, dev, previous, classes,
    output_dir=output_dir, folds=common_folds,
    emg_name="B1_rf", acc_name="B2_rf", seed=SEED,
))


**08. F4: Neural Decision-Level Late Fusion**

Select the strongest sEMG-only and acceleration-only *neural* models independently using Notebook 03's Development participant-level scores. Combine their already saved OOF soft probabilities with fixed equal weights. Preserve their identities for a final refit only if this recipe is chosen.


In [ ]:
# ================= 08. F4: Neural Decision-Level Late Fusion =================

# --------------------- 1. Participant-Grouped Model Fit ----------------------
chosen_emg, _ = select_model(
    unimodal_metrics,
    [name for name in UNIMODAL_MODELS if "_emg_" in name],
)
chosen_acc, _ = select_model(
    unimodal_metrics,
    [name for name in UNIMODAL_MODELS if "_acc_" in name],
)
print("sEMG neural component:", chosen_emg)
print("Acceleration neural component:", chosen_acc)
display(run_fixed_late_arm(
    "F4_deep_late", index, dev, previous, classes,
    output_dir=output_dir, folds=common_folds,
    emg_name=chosen_emg, acc_name=chosen_acc, seed=SEED,
))


**09. Shared Neural Fusion Protocol**

Each neural fusion architecture is trained independently on the original paired windows using seed 42, five person-disjoint Development folds, at most ten epochs, patience three, and batch size 96. Checkpoint selection uses only an inner subset of Training people. Where GPU memory allows, the normalised paired windows are cached once per fold; otherwise, the streamed path is used. This changes data throughput, not scientific inputs or model architecture.


In [ ]:
# ===================== 09. Shared Neural Fusion Protocol =====================

# --------------------- 1. Participant-Grouped Model Fit ----------------------
NEURAL_EPOCHS = 10
NEURAL_PATIENCE = 3
NEURAL_BATCH_SIZE = 96

print("Neural fusion models:", ", ".join(FUSION_MODELS))
print("Independent Development folds:", common_folds)
print("Maximum epochs:", NEURAL_EPOCHS)
print("Batch size:", NEURAL_BATCH_SIZE)


**10. F3: Dual-Branch 1D-CNN**

Two separately parameterised 1D-CNN encoders process aligned sEMG and acceleration windows, then concatenate learned embeddings for fixed-class classification. Synthetic mean replacement is a stress test on the fitted model, not real missing-sensor validation.


In [ ]:
# ======================== 10. F3: Dual-Branch 1D-CNN =========================

# --------------------- 1. Participant-Grouped Model Fit ----------------------
display(run_neural_fusion_arm(
    "F3_dual_cnn", dev, processed_dir, classes,
    output_dir=output_dir, folds=common_folds, device=device,
    seed=SEED, epochs=NEURAL_EPOCHS, patience=NEURAL_PATIENCE,
    batch_size=NEURAL_BATCH_SIZE, figure_save=save_figure,
))


**11. F3: Dual-Branch Temporal CNN**

Residual temporal convolutional encoders process each modality before a joint classification head. The same five unseen-person Development folds and Training-only early stopping govern this comparison.


In [ ]:
# ===================== 11. F3: Dual-Branch Temporal CNN ======================

# --------------------- 1. Participant-Grouped Model Fit ----------------------
display(run_neural_fusion_arm(
    "F3_dual_tcn", dev, processed_dir, classes,
    output_dir=output_dir, folds=common_folds, device=device,
    seed=SEED, epochs=NEURAL_EPOCHS, patience=NEURAL_PATIENCE,
    batch_size=NEURAL_BATCH_SIZE, figure_save=save_figure,
))


**12. F5: Gated Feature Fusion**

A learned gate combines the two temporal embeddings. The gating parameters and classification head are fitted on Training participants only. The gate's numerical weights do not constitute validated explanations.


In [ ]:
# ======================= 12. F5: Gated Feature Fusion ========================

# --------------------- 1. Participant-Grouped Model Fit ----------------------
display(run_neural_fusion_arm(
    "F5_gated", dev, processed_dir, classes,
    output_dir=output_dir, folds=common_folds, device=device,
    seed=SEED, epochs=NEURAL_EPOCHS, patience=NEURAL_PATIENCE,
    batch_size=NEURAL_BATCH_SIZE, figure_save=save_figure,
))


**13. F5: Compact Cross-Attention Fusion**

A compact two-head attention layer processes eight pooled temporal query/context tokens, with no external pretraining. Synthetic acceleration replacement is measured as an inference perturbation, not as a deployed sensor-failure test.


In [ ]:
# ================== 13. F5: Compact Cross-Attention Fusion ===================

# --------------------- 1. Participant-Grouped Model Fit ----------------------
display(run_neural_fusion_arm(
    "F5_cross_attention", dev, processed_dir, classes,
    output_dir=output_dir, folds=common_folds, device=device,
    seed=SEED, epochs=NEURAL_EPOCHS, patience=NEURAL_PATIENCE,
    batch_size=NEURAL_BATCH_SIZE, figure_save=save_figure,
))


**14. Consolidate Matched Development Fusion Evidence**

Each completed arm supplies model-specific OOF probabilities, keyed predictions and participant-level scores for all five independent Development folds. The consolidation rejects missing, incomplete or misaligned class labels and source-window identities rather than accepting partial results.

The Development-selected fusion recipe is compared against the strongest unimodal method from Notebook 03 using fixed-class macro F1 averaged equally across the same 35 people. Paired participant differences are reported without treating overlapping windows as independent participants. Reserved Test rows remain unpopulated.


In [ ]:
# ============ 14. Consolidate Matched Development Fusion Evidence ============

# ---------------- 1. Read Complete Model-Specific Outputs ---------------------
(best_unimodal, best_fusion, selection, u_summary, f_summary,
 paired_effects, robustness, fusion_metrics) = consolidate_fusion_outputs(
    index, dev, classes, unimodal_metrics, previous,
    output_dir=output_dir, folds=common_folds, seed=SEED,
    neural_settings={
        "epochs": NEURAL_EPOCHS,
        "patience": NEURAL_PATIENCE,
        "batch_size": NEURAL_BATCH_SIZE,
    },
    chosen_emg=chosen_emg, chosen_acc=chosen_acc,
)
print("Development-selected unimodal reference:", best_unimodal)
print("Development-selected fusion model:", best_fusion)
display(selection)
display(f_summary)


**15. Evaluate Additional Neural Fusion Seeds**

The strongest trained neural fusion architecture is selected using the original Development scores. It is then repeated with seeds 43 and 44 on the same person-disjoint folds. This checks training-seed variability, not generalisation to new people, and does not change the primary model selection.

Each additional seed/fold combination is saved separately. This experiment is independent of whether a classical or a neural fusion arm has the highest Development mean.


In [ ]:
# ================ 15. Evaluate Additional Neural Fusion Seeds ================

# --------------------- 1. Select Neural Fusion Finalist ----------------------
EXTRA_SEEDS = (43, 44)
best_deep, _ = select_model(
    fusion_metrics, list(FUSION_MODELS),
)
print("Selected neural fusion architecture:", best_deep)

# --------------------- 2. Repeat the Two Extra Seeds -------------------------
display(evaluate_fusion_finalist_seeds(
    best_deep, dev, processed_dir, classes,
    output_dir=output_dir, folds=common_folds, device=device,
    seeds=EXTRA_SEEDS, epochs=NEURAL_EPOCHS,
    patience=NEURAL_PATIENCE, batch_size=NEURAL_BATCH_SIZE,
))


**16. Compare Fusion Performance and Error Sensitivity**

The first figure contrasts the nine fusion candidates with the Notebook 03 unimodal benchmark using mean Development macro F1 per participant. The second shows the paired difference for every participant under the selected fusion configuration.

The third figure describes changes after **synthetic** replacement of acceleration channels with the Training mean in the fitted model. These experiments do not show observed physical sensor failures or learned missing-modality compensation. They remain Development diagnostics, not independent Test claims.


In [ ]:
# =========== 16. Compare Fusion Performance and Error Sensitivity ============

# ------------------ 1. Participant-Balanced Fusion Scores --------------------
fig, ax = plt.subplots(figsize=(11, 5), layout="constrained")
order = f_summary.iloc[::-1]
ax.barh(order.model_id, order.mean_subject_macro_f1)
reference_score = float(
    u_summary.set_index("model_id").loc[
        best_unimodal, "mean_subject_macro_f1"
    ]
)
ax.axvline(
    reference_score, color="0.35", linestyle="--",
    label=f"Unimodal reference: {best_unimodal}",
)
ax.set(
    xlabel="Mean participant macro F1",
    title="Matched Development fusion comparison",
)
ax.grid(axis="x", alpha=0.2)
ax.legend()
save_figure(fig, output_dir / "04_fusion_comparison.png")
plt.show()

# ------------------- 2. Paired Participant Differences -----------------------
matched = paired_effects.loc[
    paired_effects.model_id == best_fusion
].sort_values("subject_id")
fig, ax = plt.subplots(figsize=(12, 4), layout="constrained")
ax.bar(
    matched.subject_id, matched.fusion_minus_unimodal_macro_f1,
)
ax.axhline(0, color="0.4", linewidth=1)
ax.set(
    xlabel="Development participant",
    ylabel="Fusion minus unimodal macro F1",
    title=f"Paired participant differences: {best_fusion}",
)
ax.tick_params(axis="x", rotation=90)
ax.grid(axis="y", alpha=0.2)
save_figure(fig, output_dir / "04_subject_paired_differences.png")
plt.show()

# ------------------ 3. Synthetic Acceleration Replacement --------------------
labels = {
    "acc_missing_mean": "All acceleration replaced",
    "acc_sensor_01_missing_mean": "One accelerometer replaced",
    "acc_contiguous_gap_mean": "Contiguous acceleration interval",
}
fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")
for name, part in robustness.groupby("model_id", sort=True):
    means = part.groupby("perturbation").macro_f1.mean()
    settings = list(PERTURBATIONS)
    ax.plot(
        [labels[x] for x in settings],
        [float(means[x]) for x in settings],
        marker="o", label=name,
    )
ax.set(
    xlabel="Synthetic acceleration perturbation",
    ylabel="Mean participant macro F1",
    title="Response to mean-replaced acceleration channels",
)
ax.tick_params(axis="x", rotation=12)
ax.grid(alpha=0.2)
ax.legend()
save_figure(fig, output_dir / "04_modality_sensitivity.png")
plt.show()


**17. Research Output Files and Interpretation**

The named Notebook 04 outputs include fold-specific participant metrics, OOF prediction and probability arrays, neural training histories, synthetic perturbation scores, paired per-participant fusion differences, and the Development-only choice of the final fusion recipe.

The downloadable archive is generated outside the Git repository. It preserves the numerical evidence without adding large source arrays or model checkpoints to Git history. Notebook 05 evaluates the fixed Development-selected models on the separate reserved Test participants.


In [ ]:
# =============== 17. Research Output Files and Interpretation ================

# ------------------------- 1. Notebook Output Archive ------------------------
download_outputs_from_colab(
    "multimodal_movement_learning/output",
    f"{NOTEBOOK_NUMBER:02d}",
)
